<a href="https://colab.research.google.com/github/SardarAwais88/-ML-application-for-time-series-price-prediction/blob/main/videotranslator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install fastapi uvicorn python-multipart openai-whisper deep-translator gTTS moviepy==1.0.3 pyngrok nest-asyncio
!apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=6d29fee4e23e503c028d94dee6000964aaf1b6bfee2c434506b272d1192001e1
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages th

In [3]:
from pyngrok import ngrok
import nest_asyncio

# Replace below token with your Ngrok Authtoken
ngrok.set_auth_token("3APXK2VaJvnOrKLpBOOJl6IEfpH_4ChmnCmZPDankFQQ8c1dq")
public_url = ngrok.connect(8000).public_url
print(f"✅ Your Backend URL is: {public_url}")
# COPY THIS URL! Paste it into the Settings box on the React Frontend UI.

nest_asyncio.apply()

✅ Your Backend URL is: https://4c3b-35-227-60-29.ngrok-free.app


In [5]:
import os
import shutil
import asyncio
from fastapi import FastAPI, File, UploadFile, Form, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, FileResponse
import tempfile

# AI & Video Processing Libraries
import whisper
from deep_translator import GoogleTranslator
from gtts import gTTS
import moviepy.editor as mp

app = FastAPI(title="Free AI Video Translator Backend", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load Whisper model (We load a small model to prevent Colab from running out of RAM quickly)
# In production, use "base" or "medium" for better accuracy.
print("Loading Whisper AI Model...")
model = whisper.load_model("tiny")
print("Whisper Model Loaded Successfully!")

# Dictionary to map frontend language names/values to gTTS and deep-translator language codes
# Make sure these match what the React frontend sends
LANGUAGE_MAP = {
    "Spanish": "es",
    "es": "es",
    "French": "fr",
    "fr": "fr",
    "German": "de",
    "de": "de",
    "Hindi": "hi",
    "hi": "hi",
    "Urdu": "ur",
    "ur": "ur",
    "Arabic": "ar",
    "ar": "ar",
    "Chinese": "zh-CN",
    "Chinese (Mandarin)": "zh-CN",
    "zh": "zh-CN",
    "Japanese": "ja",
    "ja": "ja",
    "en": "en",
    "English": "en"
}

def translate_and_dub_video(video_path: str, target_lang_code: str, output_path: str):
    """
    Core AI Pipeline: Extracts audio -> Transcribes -> Translates -> TTS -> Merges back.
    """
    temp_dir = os.path.dirname(video_path)
    audio_path = os.path.join(temp_dir, "extracted_audio.wav")
    tts_audio_path = os.path.join(temp_dir, "translated_audio.mp3")

    try:
        # Step 1: Extract Audio from Video
        print("1/5: Extracting audio from video...")
        video = mp.VideoFileClip(video_path)
        if video.audio is None:
            raise Exception("No audio track found in the video to translate.")
        video.audio.write_audiofile(audio_path, logger=None)

        # Step 2: Transcribe using Whisper
        print("2/5: Transcribing audio to text...")
        result = model.transcribe(audio_path)
        original_text = result["text"]
        print(f"Transcription: {original_text}")

        # Step 3: Translate Text
        print("3/5: Translating text...")
        print(f"Target Language Code: {target_lang_code}")
        translated_text = GoogleTranslator(source='auto', target=target_lang_code).translate(original_text)
        print(f"Translated Text: {translated_text}")

        # Step 4: Text-to-Speech (Voice Generation)
        # Note: gTTS does NOT clone voices. For real voice cloning (XTTS), you'd need Coqui-TTS setup which is a massive 4GB+ dependency.
        print("4/5: Generating dubbed audio (TTS)...")
        # Ensure gTTS supports the language code. It uses simplified codes.
        gtts_lang = target_lang_code.split("-")[0] if "-" in target_lang_code and target_lang_code != "zh-CN" else target_lang_code
        if gtts_lang == "zh-CN":
            gtts_lang = "zh-CN"

        tts = gTTS(text=translated_text, lang=gtts_lang, slow=False)
        tts.save(tts_audio_path)

        # Step 5: Replace Original Audio in Video
        print("5/5: Merging new audio into video...")
        new_audio = mp.AudioFileClip(tts_audio_path)

        # We need to make sure the audio and video roughly match in length.
        # If TTS audio is longer than video, we can choose to cut it or loop video. We'll cut it.
        # Ideally, audio is speed-adjusted, but we'll do a basic overwrite here.
        final_video = video.set_audio(new_audio)

        # Write the final video
        final_video.write_videofile(
            output_path,
            codec="libx264",
            audio_codec="aac",
            temp_audiofile=os.path.join(temp_dir, "temp-audio.m4a"),
            remove_temp=True,
            logger=None
        )

        # Cleanup clips in memory
        video.close()
        new_audio.close()
        final_video.close()
        print("Pipeline Complete! Video successfully dubbed.")

    except Exception as e:
        print(f"Pipeline Error: {e}")
        # Cleanup attempting to close clips if they exist
        try: video.close()
        except: pass
        try: new_audio.close()
        except: pass
        try: final_video.close()
        except: pass
        raise e
    finally:
        # Cleanup temp audio files
        if os.path.exists(audio_path): os.remove(audio_path)
        if os.path.exists(tts_audio_path): os.remove(tts_audio_path)


@app.post("/api/translate")
async def translate_video_endpoint(file: UploadFile = File(...), language: str = Form(...)):
    if not file.filename.endswith((".mp4", ".mov")):
        return JSONResponse(status_code=400, content={"error": "Only .mp4 or .mov files are supported."})

    try:
        temp_dir = tempfile.mkdtemp()
        original_video_path = os.path.join(temp_dir, file.filename)

        with open(original_video_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)

        print(f"\n--- New Request: {file.filename} -> {language} ---")

        # Map frontend language selection to actual ISO language code
        target_lang_code = LANGUAGE_MAP.get(language)
        if not target_lang_code:
            return JSONResponse(status_code=400, content={"error": f"Unsupported language: {language}"})


        translated_video_path = original_video_path.replace(".mp4", f"_translated_{target_lang_code}.mp4").replace(".mov", f"_translated_{target_lang_code}.mp4")

        # RUN THE HEAVY AI PIPELINE
        # Note: In a production app, do NOT block the async thread with this heavy synchronous task.
        # Run it in a background task/executor. For demo simplicity, we run it directly here.
        # await asyncio.to_thread(translate_and_dub_video, original_video_path, target_lang_code, translated_video_path)

        # We run it synchronously in a separate thread so fastapi isn't blocked completely
        loop = asyncio.get_event_loop()
        await loop.run_in_executor(None, translate_and_dub_video, original_video_path, target_lang_code, translated_video_path)

        # Return the translated file
        if not os.path.exists(translated_video_path):
            raise Exception("Translated video generation failed. File not found.")

        return FileResponse(
            path=translated_video_path,
            filename=f"translated_{language}_{file.filename}",
            media_type="video/mp4"
        )

    except Exception as e:
         import traceback
         traceback.print_exc()
         return JSONResponse(status_code=500, content={"error": str(e)})

@app.get("/health")
def health_check():
    return {"status": "ok", "message": "Backend API is running!"}

if __name__ == "__main__":
    import uvicorn
    import nest_asyncio
    nest_asyncio.apply()

    # In Jupyter/Colab, standard uvicorn.run() often clashes with the running async loop.
    # The safest way to run Uvicorn inside Colab is via the Config and Server objects directly:
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)

    # Start the server gracefully inside the existing Colab event loop
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())
    print("🚀 FastAPI server is running in the background!")


Loading Whisper AI Model...
Whisper Model Loaded Successfully!
🚀 FastAPI server is running in the background!
